# DAY 05 - TASK 01: HỌC CHUYỂN GIAO (TRANSFER LEARNING) VỚI RESNET18 TRÊN CIFAR-10

### Mục tiêu thực hành:
1. **Khảo sát cơ chế Transfer Learning**: Tận dụng trọng số tiền huấn luyện (ImageNet pretrained weights) của mạng ResNet18 để chuyển giao sang phân loại bộ dữ liệu CIFAR-10 trong điều kiện dữ liệu nhỏ (2,000 ảnh).
2. **So sánh hai chế độ thích ứng**:
   - Chế độ 1 (`frozen`): Đóng băng toàn bộ backbone, chỉ huấn luyện duy nhất tầng phân loại cuối (`net.fc`).
   - Chế độ 2 (`finetune_layer4`): Mở khóa khối tích chập cuối (`net.layer4`) và tầng phân loại (`net.fc`) với learning rate phân tầng (differential learning rates).
3. **Phân tích hiệu năng và ma trận nhầm lẫn**: Đánh giá độ chính xác, đường cong học tập, ma trận nhầm lẫn và chỉ số Recall chi tiết trên từng phân lớp đối tượng.



## Bước 1: Thiết lập môi trường và cấu hình phần cứng
Import các thư viện cốt lõi, phát hiện phần cứng tính toán và xác định thư mục dữ liệu tập trung tại `data/` ở thư mục gốc dự án.



In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.libraries.seeding import seed_everything
from srcs.libraries.training import device
from srcs.libraries.models import count_parameters
from srcs.pipelines.day05_task01_transfer_learning import (
    build_resnet18,
    prepare_cifar10_loaders,
    train_resnet_model,
    collect_predictions,
    compute_confusion_matrices,
    compute_classwise_recall,
    render_learning_curves,
    render_confusion_matrices,
    CLASSES,
    DEFAULT_LR_HEAD,
    DEFAULT_LR_LAYER4
)

seed_everything(42)
comp_device = torch.device(device())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
print(f"PyTorch Version: {torch.__version__} | Device: {comp_device}")
print(f"Thư mục dữ liệu gốc: {DATA_DIR}")



## Bước 2: Khảo sát Kiến trúc mạng Pretrained ResNet18 và Cơ chế Freeze Backbone
Theo dõi số lượng tham số được tối ưu hóa giữa hai chế độ:
- **Chế độ Frozen**: Đóng băng toàn bộ các tầng tích chập, chỉ giữ `net.fc` có `requires_grad = True` (khoảng 5,130 tham số).
- **Chế độ Fine-tune Layer 4**: Mở khóa `net.layer4` và `net.fc` (khoảng 8,393,738 tham số).
- **Cố định BatchNorm**: Đưa toàn bộ các tầng `BatchNorm2d` về chế độ `eval()` để bảo toàn thống kê ImageNet.



In [ ]:
demo_frozen = build_resnet18(finetune_layer4=False, num_classes=10)
demo_finetune = build_resnet18(finetune_layer4=True, num_classes=10)

print(f"Tổng số tham số mạng ResNet18: {sum(p.numel() for p in demo_frozen.parameters()):,}")
print(f"Số tham số huấn luyện (Frozen Backbone):    {count_parameters(demo_frozen):,}")
print(f"Số tham số huấn luyện (Fine-tune Layer 4):  {count_parameters(demo_finetune):,}")



## Bước 3: Chuẩn bị dữ liệu CIFAR-10 với Tiền xử lý chuẩn ImageNet
Phân chia tập dữ liệu đảm bảo tính độc lập và tái lập:
- **Tập train**: 2,000 ảnh (batch size = 32).
- **Tập validation**: 500 ảnh (batch size = 32).
- **Phép tiền xử lý**: `Resize(256) -> CenterCrop(224) -> ToTensor() -> Normalize(ImageNet Mean, Std)`.
- Dữ liệu được lưu trữ tập trung tại `data/` ở gốc dự án.



In [ ]:
train_loader, val_loader, indices = prepare_cifar10_loaders(
    data_dir=DATA_DIR,
    train_size=2000,
    val_size=500,
    batch_size=32,
    seed=42
)
print(f"Số lượng batch huấn luyện: {len(train_loader)} (tổng 2,000 mẫu)")
print(f"Số lượng batch kiểm định:  {len(val_loader)} (tổng 500 mẫu)")
print(f"Danh sách 10 phân lớp: {CLASSES}")



## Bước 4: Huấn luyện Chế độ 1 - Frozen Backbone
Chỉ huấn luyện tầng phân loại cuối cùng `net.fc` với learning rate $10^{-3}$ qua 4 epochs.



In [ ]:
seed_everything(42)
model_frozen = build_resnet18(finetune_layer4=False, num_classes=10)
history_frozen = train_resnet_model(
    model=model_frozen,
    train_loader=train_loader,
    val_loader=val_loader,
    comp_device=comp_device,
    epochs=4,
    lr_head=DEFAULT_LR_HEAD,
    lr_layer4=None,
    mode="frozen"
)



## Bước 5: Huấn luyện Chế độ 2 - Fine-tune Layer 4
Huấn luyện đồng thời `net.layer4` (với $lr = 10^{-4}$) và `net.fc` (với $lr = 10^{-3}$) qua 4 epochs.



In [ ]:
seed_everything(42)
model_finetune = build_resnet18(finetune_layer4=True, num_classes=10)
history_finetune = train_resnet_model(
    model=model_finetune,
    train_loader=train_loader,
    val_loader=val_loader,
    comp_device=comp_device,
    epochs=4,
    lr_head=DEFAULT_LR_HEAD,
    lr_layer4=DEFAULT_LR_LAYER4,
    mode="finetune_layer4"
)



## Bước 6: Bảng so sánh tổng hợp hiệu năng giữa Frozen và Fine-tune


In [ ]:
df_summary = pd.DataFrame([
    {
        "Chế độ": "frozen",
        "Tham số train": f"{count_parameters(model_frozen):,}",
        "Train Loss": f"{history_frozen[-1]['train_loss']:.4f}",
        "Val Loss": f"{history_frozen[-1]['val_loss']:.4f}",
        "Val Accuracy": f"{history_frozen[-1]['val_acc'] * 100:.2f}%",
        "Val Acc Max": f"{max(r['val_acc'] for r in history_frozen) * 100:.2f}%",
        "Giây/epoch": f"{np.mean([r['sec_per_epoch'] for r in history_frozen]):.2f}s"
    },
    {
        "Chế độ": "finetune_layer4",
        "Tham số train": f"{count_parameters(model_finetune):,}",
        "Train Loss": f"{history_finetune[-1]['train_loss']:.4f}",
        "Val Loss": f"{history_finetune[-1]['val_loss']:.4f}",
        "Val Accuracy": f"{history_finetune[-1]['val_acc'] * 100:.2f}%",
        "Val Acc Max": f"{max(r['val_acc'] for r in history_finetune) * 100:.2f}%",
        "Giây/epoch": f"{np.mean([r['sec_per_epoch'] for r in history_finetune]):.2f}s"
    }
])
display(df_summary)



## Bước 7: Trực quan hóa Đường cong học tập (Learning Curves đối đầu)
Lưu biểu đồ vào `docs/day05/task01/image_learning_curve.png` và `bin/day05/task01/results/learning_curve.png`.



In [ ]:
curve_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day05", "task01", "image_learning_curve.png"),
    os.path.join(PROJECT_ROOT, "bin", "day05", "task01", "results", "learning_curve.png")
]
histories_dict = {
    "frozen": history_frozen,
    "finetune_layer4": history_finetune
}
render_learning_curves(histories_dict, curve_paths)
display(Image(curve_paths[0]))



## Bước 8: Trực quan hóa Ma trận nhầm lẫn và Đánh giá Recall chi tiết từng lớp
Lưu biểu đồ vào `docs/day05/task01/image_confusion_matrix.png` và `bin/day05/task01/results/confusion_matrix.png`.



In [ ]:
models_dict = {
    "frozen": model_frozen,
    "finetune_layer4": model_finetune
}
cms = compute_confusion_matrices(models_dict, val_loader, comp_device)
cm_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day05", "task01", "image_confusion_matrix.png"),
    os.path.join(PROJECT_ROOT, "bin", "day05", "task01", "results", "confusion_matrix.png")
]
render_confusion_matrices(cms, classes=CLASSES, output_paths=cm_paths)
display(Image(cm_paths[0]))

df_recall = compute_classwise_recall(cms, classes=CLASSES)
display(df_recall.round(4))



## Bước 9: Nhận xét chuyên sâu và Kết luận

1. **Hiệu năng tổng thể và Đỉnh hội tụ:**
   - Ở epoch 4, Validation Accuracy đạt 80.20% (frozen) và 80.60% (fine-tune), chênh lệch chỉ 0.40% (khoảng 2 ảnh trên 500 mẫu val).
   - Fine-tune Layer 4 hội tụ cực nhanh (đạt 78.60% ngay epoch 1) và đạt đỉnh **82.80% tại epoch 3**, cao hơn mức tốt nhất của frozen 2.6 điểm phần trăm.

2. **Dấu hiệu Overfitting của Fine-tune:**
   - Tại epoch 4, `finetune_layer4` tiếp tục giảm train loss (0.066) với train acc đạt 98.30%, nhưng val loss tăng vọt (0.481 lên 0.734) và val acc sụt giảm.
   - Nguyên nhân: Với chỉ 2,000 ảnh, việc cập nhật gần 8.4 triệu tham số khiến mạng bị dư thừa dung lượng biểu diễn và ghi nhớ mẫu huấn luyện.

3. **Tính ổn định của Frozen Backbone:**
   - Chế độ `frozen` chỉ tối ưu 5,130 tham số, train loss và val loss giảm đều đặn, độ chênh lệch rất nhỏ (train acc 84.8% vs val acc 80.2%), không có nguy cơ overfit.

4. **Đánh đổi về tài nguyên:**
   - Tham số huấn luyện chênh lệch hơn 1,600 lần, nhưng thời gian thực thi trên mỗi epoch chỉ chênh lệch ~6% (~4.10s so với ~4.36s) do cả hai đều phải thực hiện forward pass qua toàn bộ mạng ResNet18.

